# Transformação Bronze → Silver

POC de dados de energia no Microsoft Fabric, desenvolvida no âmbito da preparação para a certificação DP-700.

Este notebook lê dados base e incrementais da camada Bronze, normaliza tipos e texto, elimina duplicados, valida integridade referencial e grava tabelas Delta na camada Silver.

> Os dados utilizados são inteiramente fictícios.


## 1. Carregar bibliotecas e dados incrementais


In [ ]:
# Ler clientes incrementais (CSV)
df_clientes_inc = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("Files/Bronze/incrementais/clientes_2026_09.csv")
)

# Ler consumos incrementais (CSV)
df_consumos_inc = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("Files/Bronze/incrementais/consumos_2026_09.csv")
)

# Ler pedidos incrementais (JSON Lines)
df_pedidos_inc = spark.read.json(
    "Files/Bronze/incrementais/pedidos_2026_09.jsonl"
)

print("Clientes:")
display(df_clientes_inc)

print("Consumos:")
display(df_consumos_inc)

print("Pedidos:")
display(df_pedidos_inc)


### Validar schemas e volumes incrementais


In [ ]:
print("SCHEMA — CLIENTES")
df_clientes_inc.printSchema()
print("Total:", df_clientes_inc.count())

print("\nSCHEMA — CONSUMOS")
df_consumos_inc.printSchema()
print("Total:", df_consumos_inc.count())

print("\nSCHEMA — PEDIDOS")
df_pedidos_inc.printSchema()
print("Total:", df_pedidos_inc.count())


### Verificar duplicados nos clientes incrementais


In [ ]:
from pyspark.sql.functions import col, count

duplicados_clientes = (
    df_clientes_inc
    .groupBy("ClienteID")
    .agg(count("*").alias("Quantidade"))
    .filter(col("Quantidade") > 1)
)

display(duplicados_clientes)


## 2. Confirmar os ficheiros base da camada Bronze


In [ ]:
pastas_base = [
    "Files/Bronze/clientes",
    "Files/Bronze/consumos",
    "Files/Bronze/pedidos"
]

for pasta in pastas_base:
    print(f"\nConteúdo de {pasta}:")
    
    for ficheiro in notebookutils.fs.ls(pasta):
        print(f" - {ficheiro.name}")


## 3. Integrar, limpar e deduplicar clientes, consumos e pedidos


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# 1. LER OS DADOS-BASE
# ============================================================

df_clientes_base = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("Files/Bronze/clientes/clientes.csv")
)

df_consumos_base = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("Files/Bronze/consumos/consumos.csv")
)

df_pedidos_base = spark.read.json(
    "Files/Bronze/pedidos/pedidos.jsonl"
)

# ============================================================
# 2. JUNTAR BASE + INCREMENTAIS
# ============================================================

df_clientes_total = df_clientes_base.unionByName(
    df_clientes_inc,
    allowMissingColumns=True
)

df_consumos_total = df_consumos_base.unionByName(
    df_consumos_inc,
    allowMissingColumns=True
)

df_pedidos_total = df_pedidos_base.unionByName(
    df_pedidos_inc,
    allowMissingColumns=True
)

# ============================================================
# 3. LIMPEZA E CONVERSÃO DOS TIPOS
# ============================================================

df_clientes_silver = (
    df_clientes_total
    .withColumn("ClienteID", F.trim(F.col("ClienteID")))
    .withColumn("NomeCliente", F.trim(F.col("NomeCliente")))
    .withColumn("Concelho", F.trim(F.col("Concelho")))
    .withColumn("Segmento", F.trim(F.col("Segmento")))
    .withColumn("EstadoCliente", F.trim(F.col("EstadoCliente")))
    .withColumn("DataAdesao", F.to_date("DataAdesao"))
    .withColumn(
        "UltimaAtualizacao",
        F.to_timestamp("UltimaAtualizacao")
    )
    .filter(F.col("ClienteID").isNotNull())
)

df_consumos_silver = (
    df_consumos_total
    .withColumn("ConsumoID", F.trim(F.col("ConsumoID")))
    .withColumn("UnidadeID", F.trim(F.col("UnidadeID")))
    .withColumn("DataLeitura", F.to_date("DataLeitura"))
    .withColumn("Periodo", F.to_timestamp("Periodo"))
    .withColumn(
        "QuantidadeKWh",
        F.col("QuantidadeKWh").cast("double")
    )
    .withColumn("TipoLeitura", F.trim(F.col("TipoLeitura")))
    .withColumn(
        "DataIngestao",
        F.to_timestamp("DataIngestao")
    )
    .filter(F.col("ConsumoID").isNotNull())
)

df_pedidos_silver = (
    df_pedidos_total
    .withColumn("PedidoID", F.trim(F.col("PedidoID")))
    .withColumn("UnidadeID", F.trim(F.col("UnidadeID")))
    .withColumn("Canal", F.trim(F.col("Canal")))
    .withColumn("Estado", F.trim(F.col("Estado")))
    .withColumn("Prioridade", F.trim(F.col("Prioridade")))
    .withColumn("Servico", F.trim(F.col("Servico")))
    .withColumn(
        "DataAbertura",
        F.to_timestamp("DataAbertura")
    )
    .withColumn(
        "DataConclusao",
        F.to_timestamp("DataConclusao")
    )
    .withColumn(
        "UltimaAtualizacao",
        F.to_timestamp("UltimaAtualizacao")
    )
    .filter(F.col("PedidoID").isNotNull())
)

# ============================================================
# 4. DEDUPLICAÇÃO — CONSERVAR O REGISTO MAIS RECENTE
# ============================================================

janela_clientes = (
    Window
    .partitionBy("ClienteID")
    .orderBy(F.col("UltimaAtualizacao").desc_nulls_last())
)

df_clientes_silver = (
    df_clientes_silver
    .withColumn("_numero_linha", F.row_number().over(janela_clientes))
    .filter(F.col("_numero_linha") == 1)
    .drop("_numero_linha")
)

janela_consumos = (
    Window
    .partitionBy("ConsumoID")
    .orderBy(F.col("DataIngestao").desc_nulls_last())
)

df_consumos_silver = (
    df_consumos_silver
    .withColumn("_numero_linha", F.row_number().over(janela_consumos))
    .filter(F.col("_numero_linha") == 1)
    .drop("_numero_linha")
)

janela_pedidos = (
    Window
    .partitionBy("PedidoID")
    .orderBy(F.col("UltimaAtualizacao").desc_nulls_last())
)

df_pedidos_silver = (
    df_pedidos_silver
    .withColumn("_numero_linha", F.row_number().over(janela_pedidos))
    .filter(F.col("_numero_linha") == 1)
    .drop("_numero_linha")
)

# ============================================================
# 5. GRAVAR COMO TABELAS DELTA
# ============================================================

(
    df_clientes_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_clientes")
)

(
    df_consumos_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_consumos")
)

(
    df_pedidos_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_pedidos")
)

print("Tabelas Silver criadas com sucesso.")


### Validar as primeiras tabelas Silver


In [ ]:
print("Clientes Silver:", spark.table("silver_clientes").count())
print("Consumos Silver:", spark.table("silver_consumos").count())
print("Pedidos Silver:", spark.table("silver_pedidos").count())

display(spark.table("silver_clientes"))


## 4. Processar unidades e faturas


In [ ]:
pastas_restantes = [
    "Files/Bronze/unidades",
    "Files/Bronze/faturas"
]

for pasta in pastas_restantes:
    print(f"\nConteúdo de {pasta}:")
    
    for ficheiro in notebookutils.fs.ls(pasta):
        print(f" - {ficheiro.name}")


In [ ]:
# Ler Unidades
df_unidades = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("Files/Bronze/unidades/unidades_consumo.csv")
)

# Ler Faturas
df_faturas = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("Files/Bronze/faturas/faturas.csv")
)

print("SCHEMA — UNIDADES")
df_unidades.printSchema()
print("Total:", df_unidades.count())

print("\nSCHEMA — FATURAS")
df_faturas.printSchema()
print("Total:", df_faturas.count())

print("\nAmostra — UNIDADES")
display(df_unidades.limit(10))

print("\nAmostra — FATURAS")
display(df_faturas.limit(10))


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# 1. LIMPAR UNIDADES
# ============================================================

df_unidades_silver = (
    df_unidades
    .withColumn("UnidadeID", F.trim(F.col("UnidadeID")))
    .withColumn("ClienteID", F.trim(F.col("ClienteID")))
    .withColumn("TipoUnidade", F.trim(F.col("TipoUnidade")))
    .withColumn("Concelho", F.trim(F.col("Concelho")))
    .withColumn("EstadoUnidade", F.trim(F.col("EstadoUnidade")))
    .withColumn("DataAtivacao", F.to_date("DataAtivacao"))
    .withColumn(
        "UltimaAtualizacao",
        F.to_timestamp("UltimaAtualizacao")
    )
    .filter(F.col("UnidadeID").isNotNull())
    .filter(F.col("ClienteID").isNotNull())
)

# Conservar a versão mais recente de cada unidade
janela_unidades = (
    Window
    .partitionBy("UnidadeID")
    .orderBy(F.col("UltimaAtualizacao").desc_nulls_last())
)

df_unidades_silver = (
    df_unidades_silver
    .withColumn("_numero_linha", F.row_number().over(janela_unidades))
    .filter(F.col("_numero_linha") == 1)
    .drop("_numero_linha")
)

# ============================================================
# 2. LIMPAR FATURAS
# ============================================================

df_faturas_silver = (
    df_faturas
    .withColumn("FaturaID", F.trim(F.col("FaturaID")))
    .withColumn("ClienteID", F.trim(F.col("ClienteID")))
    .withColumn("Periodo", F.to_timestamp("Periodo"))
    .withColumn("DataEmissao", F.to_date("DataEmissao"))
    .withColumn("DataVencimento", F.to_date("DataVencimento"))
    .withColumn("DataPagamento", F.to_date("DataPagamento"))
    .withColumn(
        "ValorSemIVA",
        F.col("ValorSemIVA").cast("decimal(12,2)")
    )
    .withColumn(
        "ValorIVA",
        F.col("ValorIVA").cast("decimal(12,2)")
    )
    .withColumn(
        "ValorTotal",
        F.col("ValorTotal").cast("decimal(12,2)")
    )
    .withColumn(
        "EstadoPagamento",
        F.trim(F.col("EstadoPagamento"))
    )
    .withColumn(
        "UltimaAtualizacao",
        F.to_timestamp("UltimaAtualizacao")
    )
    .filter(F.col("FaturaID").isNotNull())
    .filter(F.col("ClienteID").isNotNull())
)

# Conservar a versão mais recente de cada fatura
janela_faturas = (
    Window
    .partitionBy("FaturaID")
    .orderBy(F.col("UltimaAtualizacao").desc_nulls_last())
)

df_faturas_silver = (
    df_faturas_silver
    .withColumn("_numero_linha", F.row_number().over(janela_faturas))
    .filter(F.col("_numero_linha") == 1)
    .drop("_numero_linha")
)

# ============================================================
# 3. GRAVAR AS TABELAS DELTA
# ============================================================

(
    df_unidades_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_unidades")
)

(
    df_faturas_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_faturas")
)

print("Tabelas silver_unidades e silver_faturas criadas.")


## 5. Validar integridade referencial


In [ ]:
clientes = spark.table("silver_clientes")
unidades = spark.table("silver_unidades")
faturas = spark.table("silver_faturas")

# Registos sem Cliente correspondente
unidades_sem_cliente = (
    unidades
    .join(clientes, "ClienteID", "left_anti")
)

faturas_sem_cliente = (
    faturas
    .join(clientes, "ClienteID", "left_anti")
)

print("Unidades Silver:", unidades.count())
print("Faturas Silver:", faturas.count())
print("Unidades sem cliente:", unidades_sem_cliente.count())
print("Faturas sem cliente:", faturas_sem_cliente.count())


In [ ]:
display(
    faturas_sem_cliente.select(
        "FaturaID",
        "ClienteID",
        "DataEmissao",
        "ValorTotal",
        "EstadoPagamento"
    )
)


## 6. Separar faturas sem cliente correspondente


In [ ]:
# Chaves válidas de clientes
clientes_validos = (
    spark.table("silver_clientes")
    .select("ClienteID")
    .distinct()
)

# Faturas com cliente válido
df_faturas_validas = (
    df_faturas_silver
    .join(clientes_validos, "ClienteID", "left_semi")
)

# Faturas sem cliente correspondente
df_faturas_rejeitadas = (
    df_faturas_silver
    .join(clientes_validos, "ClienteID", "left_anti")
    .withColumn(
        "MotivoRejeicao",
        F.lit("ClienteID inexistente em silver_clientes")
    )
    .withColumn(
        "DataRejeicao",
        F.current_timestamp()
    )
)

# Gravar os registos rejeitados para análise
(
    df_faturas_rejeitadas.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_faturas_rejeitadas")
)

# Substituir Silver por faturas válidas
(
    df_faturas_validas.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_faturas")
)

print("Faturas válidas:", df_faturas_validas.count())
print("Faturas rejeitadas:", df_faturas_rejeitadas.count())


## 7. Validação final da camada Silver


In [ ]:
print(
    "Faturas válidas:",
    spark.table("silver_faturas").count()
)

print(
    "Faturas rejeitadas:",
    spark.table("silver_faturas_rejeitadas").count()
)

display(
    spark.table("silver_faturas_rejeitadas")
)
